## Harmony Py Library
### Batch Processing Example

This notebook demonstrates submitting many Harmony requests at once and processing them
as a batch, without having to track each job's completion individually. This is useful
when you have a large number of independent requests (e.g. one per granule, date range, or
region of interest) that you want to send quickly and collect results from once they're all
done.

Three methods make this possible:

* `submit_batch()` - submits a list of `Request` objects concurrently and returns a list of
  job IDs, in the same order as the requests.
* `wait_for_batch()` - waits for a list of job IDs to each reach a terminal state, polling
  them concurrently, and reports back which jobs succeeded and which failed.
* `download_batch()` - downloads the output files for a list of job IDs concurrently, or with
  `combine_csv`, combines the jobs' CSV outputs into a single CSV file.

In [ ]:
import sys
import helper
helper.install_project_and_dependencies('..')

import datetime as dt
from harmony import BBox, Client, Collection, Request, Environment

#### Build a batch of requests

Here we build several independent subset requests against the same collection, each for a
different time range. In practice these might instead vary by spatial area, granule, or any
other criteria.

In [ ]:
harmony_client = Client(env=Environment.UAT)  # assumes .netrc usage

collection = Collection(id='C1234088182-EEDTEST')

time_ranges = [
    {
        'start': dt.datetime(2020, 1, 1),
        'stop': dt.datetime(2020, 1, 5),
    },
    {
        'start': dt.datetime(2020, 1, 1),
        'stop': dt.datetime(2020, 1, 5),
    },
    {
        'start': dt.datetime(2020, 2, 1),
        'stop': dt.datetime(2020, 2, 5),
    },
]

requests = [Request(collection=collection, temporal=time_range, labels=['batch_submit']) for time_range in time_ranges]

##### Add a request that we know will fail due to an access issue downloading the file.

In [ ]:
failing_request = Request(
    collection=Collection(id='C1234724470-POCLOUD'),
    spatial=BBox(-45,0,45,90),
    granule_id='G1236546414-POCLOUD'
)
requests.append(failing_request)

#### Submit the batch

`submit_batch()` submits every request concurrently rather than waiting for each one to be
accepted before submitting the next, and returns the resulting job IDs in the same order as
`requests`.

In [ ]:
job_ids = harmony_client.submit_batch(requests)
job_ids

#### Wait for the batch to finish

`wait_for_batch()` polls every job in the batch concurrently until each one reaches a
terminal state (`successful`, `failed`, `canceled`, `complete_with_errors`, or `paused`), and
returns a `BatchStatus` with a count and a list of job IDs for each status.

In [ ]:
batch_status = harmony_client.wait_for_batch(job_ids, show_progress=True)
batch_status.counts

##### Get the list of successful job ids.

In [ ]:
batch_status.job_ids["successful"]

#### Inspect any failures

For jobs that didn't succeed, `status()` can be used to see why.

In [ ]:
for job_id in batch_status.job_ids['failed']:
    status = harmony_client.status(job_id)
    print(job_id, status['status'], status['message'])

#### Download the outputs

`download_batch()` downloads the output files for every job in the batch concurrently and
returns a dict mapping each job ID to a list of `Future`s, one per output file. Calling
`.result()` on a `Future` blocks until that particular file finishes downloading and returns
its local filename.

It's safe to pass every job ID from the batch, regardless of status — a job with no output
files (e.g. one that failed or has no results) just maps to an empty list.

In [ ]:
downloads = harmony_client.download_batch(job_ids, directory='/tmp', overwrite=True)

for job_id, futures in downloads.items():
    filenames = [f.result() for f in futures]
    print(f'{job_id}: {filenames}')

### OB.DAAC Satellite In-Situ Matchup Batch

OB.DAAC's `satellite-in-situ-matchup` service (SeaDAS) extracts the satellite pixels around an
in-situ measurement and returns them as a CSV file. A typical matchup workflow sends one small
request per in-situ measurement: a box of about 0.5° around the measurement location and a one hour
window centered on the measurement time, which makes it a good fit for batch processing.

The requests below cover several OB.DAAC Level-2 ocean color collections, one request each.

In [ ]:
harmony_client = Client(env=Environment.UAT)

matchups = [
    {   # VIIRSN_L2_OC: Suomi-NPP VIIRS Level-2 Regional Ocean Color
        'collection': 'C1272968279-OB_CLOUD',
        'bbox': BBox(40.287, 42.675, 40.787, 43.175),
        'start': dt.datetime(2017, 4, 6, 9, 3),
        'stop': dt.datetime(2017, 4, 6, 10, 3),
    },
    {   # PACE_OCI_L2_AOP_NRT: PACE OCI Level-2 Regional Apparent Optical Properties NRT
        'collection': 'C1281448693-OB_CLOUD',
        'bbox': BBox(18.765, 55.664, 19.265, 56.164),
        'start': dt.datetime(2026, 6, 3, 11, 5),
        'stop': dt.datetime(2026, 6, 3, 12, 5),
    },
    {   # MODIST_L2_OC: Terra MODIS Level-2 Regional Ocean Color
        'collection': 'C1272944705-OB_CLOUD',
        'bbox': BBox(-119.75, 32.75, -119.25, 33.25),
        'start': dt.datetime(2010, 10, 10, 17, 37, 30),
        'stop': dt.datetime(2010, 10, 10, 18, 37, 30),
    },
    {   # VIIRSJ2_L2_OC: NOAA-21 VIIRS Level-2 Regional Ocean Color
        'collection': 'C1276697749-OB_CLOUD',
        'bbox': BBox(44.75, -30.25, 45.25, -29.75),
        'start': dt.datetime(2023, 6, 8, 10, 39),
        'stop': dt.datetime(2023, 6, 8, 11, 39),
    },
    {   # VIIRSJ1_L2_OC: NOAA-20 VIIRS Level-2 Regional Ocean Color
        'collection': 'C1273019288-OB_CLOUD',
        'bbox': BBox(24.75, 33.75, 25.25, 34.25),
        'start': dt.datetime(2019, 4, 6, 10, 9),
        'stop': dt.datetime(2019, 4, 6, 11, 9),
    },
    {   # MODISA_L2_OC_NRT: Aqua MODIS Level-2 Regional Ocean Color NRT
        'collection': 'C1272940552-OB_CLOUD',
        'bbox': BBox(57.75, 14.75, 58.25, 15.25),
        'start': dt.datetime(2026, 6, 3, 11, 22, 30),
        'stop': dt.datetime(2026, 6, 3, 12, 22, 30),
    },
]

obdaac_requests = [
    Request(
        collection=Collection(id=matchup['collection']),
        spatial=matchup['bbox'],
        temporal={'start': matchup['start'], 'stop': matchup['stop']},
        format='text/csv',
        variables=['all'],
        labels=['batch_submit'],
    )
    for matchup in matchups
]

#### Submit the matchup batch and wait for it to finish

In [ ]:
obdaac_job_ids = harmony_client.submit_batch(obdaac_requests)
obdaac_batch_status = harmony_client.wait_for_batch(obdaac_job_ids, show_progress=True)
obdaac_batch_status.counts

#### Download the results as a single CSV file

Each matchup job produces a CSV file with a row per in-situ measurement, holding the
measurement's location and time, the matched satellite granule, whether the match was
accepted (`matchup_status`), and statistics of the satellite pixels around it.

Passing `combine_csv` to `download_batch()` downloads every job's CSV output and combines them
into a single CSV file. The files don't need identical columns (different sensors report
different bands): the combined file has every column from any of them, left empty in rows that
don't have it. `include_job_id=True` adds a `job_id` column as the combined file's first
column, holding the ID of the job each row came from, so each row can be traced back to its
request.

Instead of the per-job Futures, `download_batch()` returns a `CombinedCsvResult` with the
combined file's `filename` and `job_ids_without_outputs`: the jobs (e.g. ones that failed) that
had no output files and so have no rows in the combined file. Those job IDs are also printed as
a warning.

In [ ]:
import csv

combined = harmony_client.download_batch(
    obdaac_job_ids,
    directory='/tmp',
    overwrite=True,
    combine_csv='obdaac_matchups.csv',
    include_job_id=True,
)
print(f'Jobs without outputs: {combined.job_ids_without_outputs}')

with open(combined.filename, newline='') as csv_file:
    reader = csv.DictReader(csv_file)
    rows = list(reader)
print(f'{combined.filename}: {len(rows)} rows, {len(reader.fieldnames)} columns')
for row in rows:
    print(f"    {row['job_id']}: {row['obs_lat']}, {row['obs_lon']}, {row['obs_datetime']}, "
          f"{row['granule']}, {row['matchup_status']}")

#### Combining a batch whose outputs aren't all CSV files

`combine_csv` only works when every output in the batch is a CSV file. Here we build a batch
with two of the matchup requests above plus a request whose output is a GeoTIFF.

In [ ]:
non_csv_request = Request(
    collection=Collection(id='C1233800302-EEDTEST'),
    granule_id=['G1233800352-EEDTEST'],
    spatial=BBox(-10, 0, 10, 20),
    format='image/tiff',
    labels=['batch_submit'],
)
mixed_requests = obdaac_requests[:2] + [non_csv_request]

mixed_job_ids = harmony_client.submit_batch(mixed_requests)
mixed_batch_status = harmony_client.wait_for_batch(mixed_job_ids, show_progress=True)
mixed_batch_status.counts

All three jobs succeed, but asking `download_batch()` to combine their outputs raises a
`ValueError` naming the outputs that aren't CSV files. No combined file is written, and the
individual outputs downloaded along the way are cleaned up, so nothing is left in `directory`.

To get the CSV outputs combined anyway, pass only the CSV jobs' IDs to `download_batch()` with
`combine_csv`, and download the other jobs without it.

In [ ]:
import os

mixed_filename = '/tmp/obdaac_mixed_matchups.csv'

try:
    harmony_client.download_batch(
        mixed_job_ids, directory='/tmp', overwrite=True, combine_csv='obdaac_mixed_matchups.csv'
    )
except ValueError as e:
    print(f'ValueError: {e}')

print(f'{mixed_filename} exists: {os.path.exists(mixed_filename)}')